# Case 2 — multivariate wPCA on Carajas TMI + radiometric U

This notebook runs the paper's **Case 2**: two variables (TMI and radiometric uranium) combined by **score concatenation**, reference deposit **Alemao (Deposit 3)**. Each variable is decomposed on its own; the standardized principal-component scores are concatenated; and windows are ranked by weighted distance to the reference in that combined score space. It uses the current config-driven pipeline, the same path as the README quickstart.

(See `01_carajas_univariate_demo.ipynb` for the plain-language description of what wPCA does.)

**Prerequisites:** the environment from `requirements.txt`, and **both** Carajas data folders (TMI and radiometric U) placed under `data/` (see the README 'Public Data Download' section).

## 1. Set up paths

In [ ]:
import os, sys
from pathlib import Path

# This notebook lives in <repo>/notebooks/. Find the repository root and put
# the source package on the path, then work from the repo root so the config's
# relative data paths resolve.
REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('Repository root:', REPO)

## 2. Check the input data is present

In [ ]:
# Case 2 needs both Carajas data folders (TMI and radiometric U).
uni = REPO / 'data' / 'Carajas_Brazil_Univariate_TMI'
multi = REPO / 'data' / 'Carajas_Brazil_Multivariate_TMI_U'
missing = [d for d in (uni, multi) if not any(d.glob('*.ers'))]
if missing:
    for d in missing:
        print('Missing data folder:', d)
    print('Download the two public Drive folders linked in the README and place them under data/.')
else:
    print('Data found:', uni.name, 'and', multi.name)

## 3. Run the multivariate wPCA workflow

This takes about a minute. It concatenates the two variables' PC scores and ranks the windows against the Alemao reference, writing the top windows, figures, and recovery curve to a fresh `outputs/` folder.

In [ ]:
from spatial_pca.pipeline import run_spca_from_config

# Case 2: multivariate TMI + radiometric U, score concatenation, reference deposit
# Alemao (Deposit 3). Each variable's standardized PC scores are concatenated and
# windows are ranked in the combined score space. Same run as the README quickstart
#   python scripts/run_project_from_config.py \
#     --config configs/carajas_multi_tmi_u_concat_scores_tmi17_u25.yaml --deposit 3
results = run_spca_from_config(
    REPO / 'configs' / 'carajas_multi_tmi_u_concat_scores_tmi17_u25.yaml',
    deposit_1based=3,
    top_k=250,
)
res = results[0]
out = Path(res.top_windows_path).parent
print('Output folder :', out)
print('Top windows   :', res.top_windows_path)
print('Recovery plot :', res.recovery_plot_path)

## 4. Show the results

In [ ]:
from IPython.display import Image, display

# Prediction map: the top-ranked windows over the TMI grid.
for p in sorted(out.glob('*Top_*Predicted_Windows.png')):
    display(Image(filename=str(p)))

# Cumulative footprint-recovery curve vs a random selection.
rec = out / 'cumulative_footprint_recovery_fraction.png'
if rec.exists():
    display(Image(filename=str(rec)))

## Expected headline result

In the top 250 windows this run reports about **63% cumulative footprint recovery** with **3 test deposits hit**, with Alemao as the reference. The paper's exact Case 2 headline numbers (TMI k = 2, U k = 8, balance weight set from the other deposits' univariate performance: **63.7% recovery, AUC 106.1, 3 of 4 test deposits**) are reproduced by the scientific-record script:

```bash
python paper/run_ablation_checks.py
```

> Note: `configs/carajas_multi_tmi_u.yaml` is the full 5-deposit x 20-k selection sweep behind Appendix B — it is much slower and expects the univariate sweep tables, so it is not the place to start. Use `carajas_multi_tmi_u_concat_scores_tmi17_u25.yaml` (above) for the Case 2 demo.